# Tunable Coupler Sweeps

## Setup
### Enviornment Setup
Below we setup the computational resources used in the sweep. No physically meaningful quantities should be defined here.

The below cell finds the root directory of the package and binds its fully qualified path to to `ROOT`. The directories for figures and outputs are then bound to the corresponding variables. Finally, we ensure that the julia enviornment is set to the env of the Xonium project, who's `.toml` specification files are in `ROOT`.

The imported packages are then guaranteed to exist, given that the correct julia env is loaded.

In [ ]:
using ProjectRoot
using Pkg
const ROOT = @projectroot 
const OUTPUT = @projectroot "output"
const FIGURES = @projectroot "figures"
Pkg.activate(ROOT)

using Revise
using QuantumToolbox
using XoniumUnitCell
using Base.Iterators: product
using StructArrays
using ProgressMeter
using CairoMakie

  Activating project at `~/Projects/xonium/XoniumUnitCell.jl`


Multithreading setup: `BLAS` threads must be pinned to 1. The `BLAS` and `LAPACK` fortran linear-algebra backends will default to using all threads otherwise. This may cause CPU thrashing when we try to multithread over the sweeps.

In [ ]:
using LinearAlgebra 

LinearAlgebra.BLAS.set_num_threads(1)


# everything below is simply checking that computational
# resources are not over-allocated, or otherwise displaying 
# the resources available on the device.
logical_cpu_count = Sys.CPU_THREADS
session_thread_count = Threads.nthreads()
spare_threads = logical_cpu_count - session_thread_count
if spare_threads < 0 
    @warn "More threads allocated to Julia session \n than available on device" logical_cpu_count session_thread_count
elseif spare_threads == 0 
    @warn "At least one logical CPU core \n should be left free for system processes"
else
    @info "Total threads used by this Julia session: $(Threads.nthreads())"
    @info "Total logical CPUs available on this device: $(Sys.CPU_THREADS)"
end

[ Info: Total threads used by this Julia session: 14
[ Info: Total logical CPUs available on this device: 16
┌ Info: 
│   CPU_MODEL_NAMES =
│    16-element Vector{String}:
│     "Intel(R) Core(TM) Ultra 9 285H"
│     "Intel(R) Core(TM) Ultra 9 285H"
│     "Intel(R) Core(TM) Ultra 9 285H"
│     "Intel(R) Core(TM) Ultra 9 285H"
│     "Intel(R) Core(TM) Ultra 9 285H"
│     ⋮
│     "Intel(R) Core(TM) Ultra 9 285H"
│     "Intel(R) Core(TM) Ultra 9 285H"
│     "Intel(R) Core(TM) Ultra 9 285H"
│     "Intel(R) Core(TM) Ultra 9 285H"
└     "Intel(R) Core(TM) Ultra 9 285H"
